# Hypersim Dataset and Sampler Test

In [ ]:
import os
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from open_vocab_mot.data.hypersim_ds import HypersimVideoReIDDataset, HypersimBatchIterableDataset
from open_vocab_mot.definitions import HYPERSIM_DATASET_PATH, HYPERSIM_DATASET_SIDECAR_PATH

In [ ]:
print("Loading dataset...")
ds = HypersimVideoReIDDataset(
    ds_root=HYPERSIM_DATASET_PATH,
    sidecar_path=HYPERSIM_DATASET_SIDECAR_PATH,
    min_frames_per_sequence=4,
    load_image_pil=True,  # Load PIL images for easy visualization
    load_image_tensor=False,
    crop_to_object=True,
    crop_padding=50,
    apply_mask=False
)

print(f"Loaded {len(ds.global_identities)} identities across {len(ds.scene_to_global_ids)} scenes.")


In [ ]:
print("Creating iterable dataset sampler...")
iterable_ds = HypersimBatchIterableDataset(
    dataset=ds,
    batches_per_epoch=100,
    num_identities_per_batch=2,
    num_hard_negatives_per_positive=1,
    num_sequences_per_tracklet=2,
    num_frames_per_sequence=8,
    seed=42
)

In [ ]:
iterator = iter(iterable_ds)

In [ ]:
print("Sampling a batch...")

batch = next(iterator)
print(f"Batch has {len(batch)} items.")

# Group by identity and sequence
from collections import defaultdict
grouped = defaultdict(lambda: defaultdict(list))

for item in batch:
    grouped[item.identity_id][item.sequence_id].append(item)

print(f"Grouped into {len(grouped)} identities.")

## Visualization
The cell below plots the sequences grouped by identity. You should see different sequences for the same identity being sampled from different cameras (Hard Positives) if they exist, and other identities with the same semantic label (Hard Negatives).

In [ ]:
# Visualize the batch
for identity_id, sequences in grouped.items():
    identity_info = ds.global_identities[identity_id]
    print(f"Identity: {identity_id} | Original Obj ID: {identity_info.original_obj_id} | Semantic Label: {identity_info.semantic_label}")
    
    # Calculate how many frames are actually in these sequences
    # We can just look at the first sequence's length
    num_frames = len(list(sequences.values())[0])
    
    fig, axes = plt.subplots(len(sequences), num_frames, figsize=(4 * num_frames, 4 * len(sequences)))
    if len(sequences) == 1:
        axes = [axes]
        
    for idx, (seq_id, items) in enumerate(sequences.items()):
        for f_idx, item in enumerate(items):
            # Handle 1D array case if there's only 1 sequence
            if len(sequences) == 1:
                ax = axes[0][f_idx]
            else:
                ax = axes[idx][f_idx]
                
            ax.imshow(item.frame)
            cam_name = item.frame_id.split("_")[0] + "_" + item.frame_id.split("_")[1]
            ax.set_title(f"Seq: {seq_id} | Cam: {cam_name}")
            ax.axis("off")
            
    plt.tight_layout()
    plt.show()
